## AUDITORIA DA HARMONIZAÇÃO TERRITORIAL

In [2]:
# ============================================================
# AUDITORIA DA HARMONIZAÇÃO TERRITORIAL
# ============================================================
from _setup import *
import unicodedata
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from enem_pipeline.config import CONSUMO_DIR, DATA_DIR

import pandas as pd
import geopandas as gpd

UF = "PA"

# ------------------------------------------------------------
# 1. Carregar dados originais
# ------------------------------------------------------------

consumo_dir = CONSUMO_DIR / f"uf={UF}"

arquivos = {
    "painel": consumo_dir / "painel_municipal.parquet",
    "faixa": consumo_dir / "faixa_etaria_municipal.parquet",
    "grupo": consumo_dir / "desempenho_grupos_municipal.parquet",
}

arquivo_shape = (
    DATA_DIR
    / "shapefiles"
    / f"{UF}_Municipios_2025"
    / f"{UF}_Municipios_2025.shp"
)

dados = {
    nome: pd.read_parquet(caminho)
    for nome, caminho in arquivos.items()
}

shape = gpd.read_file(arquivo_shape)

# ------------------------------------------------------------
# 2. Verificar tipos e valores dos códigos
# ------------------------------------------------------------

print("=" * 80)
print("1. TIPOS DOS CÓDIGOS MUNICIPAIS")
print("=" * 80)

print("\nSHAPEFILE")
print("CD_MUN:", shape["CD_MUN"].dtype)
print("Exemplos:", shape["CD_MUN"].dropna().head().tolist())

for nome, df in dados.items():
    print(f"\n{nome.upper()}")
    print("CO_MUNICIPIO_TERRITORIO:",
          df["CO_MUNICIPIO_TERRITORIO"].dtype)
    print(
        "Exemplos:",
        df["CO_MUNICIPIO_TERRITORIO"].dropna().head().tolist()
    )

# ------------------------------------------------------------
# 3. Verificar integridade dos códigos no shapefile
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. INTEGRIDADE DO SHAPEFILE")
print("=" * 80)

print("Total de registros:", len(shape))
print("Códigos nulos:", shape["CD_MUN"].isna().sum())
print(
    "Códigos duplicados:",
    shape["CD_MUN"].duplicated().sum()
)
print(
    "Nomes municipais nulos:",
    shape["NM_MUN"].isna().sum()
)

# ------------------------------------------------------------
# 4. Verificar correspondência entre os códigos
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. CORRESPONDÊNCIA ENTRE PARQUETS E SHAPEFILE")
print("=" * 80)

codigos_shape = set(shape["CD_MUN"].dropna())

for nome, df in dados.items():

    codigos = df["CO_MUNICIPIO_TERRITORIO"].dropna()

    codigos_unicos = set(codigos.unique())

    encontrados = codigos_unicos.intersection(codigos_shape)
    ausentes = codigos_unicos.difference(codigos_shape)

    print(f"\n{nome.upper()}")
    print("Códigos únicos:", len(codigos_unicos))
    print("Correspondências exatas:", len(encontrados))
    print("Sem correspondência:", len(ausentes))

    if ausentes:
        print("Exemplos sem correspondência:", list(ausentes)[:10])

# ------------------------------------------------------------
# 5. Executar update_nomes e validar resultado
# ------------------------------------------------------------

from utils.features_names import update_nomes

print("\n" + "=" * 80)
print("4. VALIDAÇÃO DO UPDATE_NOMES")
print("=" * 80)

for nome, df_original in dados.items():

    df_teste = df_original.copy()

    resultado = update_nomes(df_teste, shape.copy())

    print(f"\n{nome.upper()}")
    print("Registros:", len(resultado))
    print(
        "Nomes nulos após atualização:",
        resultado["NO_MUNICIPIO_TERRITORIO"].isna().sum()
    )
    print(
        "Municípios únicos:",
        resultado["CO_MUNICIPIO_TERRITORIO"].nunique()
    )
    print(
        "Nomes únicos:",
        resultado["NO_MUNICIPIO_TERRITORIO"].nunique()
    )

    print(
        resultado[
            [
                "CO_MUNICIPIO_TERRITORIO",
                "NO_MUNICIPIO_TERRITORIO"
            ]
        ].drop_duplicates().head(10).to_string(index=False)
    )

print("\nAUDITORIA CONCLUÍDA.")

1. TIPOS DOS CÓDIGOS MUNICIPAIS

SHAPEFILE
CD_MUN: str
Exemplos: ['1507706', '1507755', '1507102', '1506906', '1507003']

PAINEL
CO_MUNICIPIO_TERRITORIO: string
Exemplos: ['1500800', '1501402', '1502707', '1504422', '1505502']

FAIXA
CO_MUNICIPIO_TERRITORIO: string
Exemplos: ['1500800', '1500800', '1500800', '1500800', '1500800']

GRUPO
CO_MUNICIPIO_TERRITORIO: string
Exemplos: ['1500800', '1500800', '1500800', '1500800', '1500800']

2. INTEGRIDADE DO SHAPEFILE
Total de registros: 144
Códigos nulos: 0
Códigos duplicados: 0
Nomes municipais nulos: 0

3. CORRESPONDÊNCIA ENTRE PARQUETS E SHAPEFILE

PAINEL
Códigos únicos: 143
Correspondências exatas: 143
Sem correspondência: 0

FAIXA
Códigos únicos: 143
Correspondências exatas: 143
Sem correspondência: 0

GRUPO
Códigos únicos: 143
Correspondências exatas: 143
Sem correspondência: 0

4. VALIDAÇÃO DO UPDATE_NOMES

PAINEL
Registros: 2412
Nomes nulos após atualização: 0
Municípios únicos: 143
Nomes únicos: 143
CO_MUNICIPIO_TERRITORIO NO_MUNICI